In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder,OneHotEncoder
from sklearn.pipeline import Pipeline
from keras_tuner.tuners import RandomSearch
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
import pickle




In [16]:
data = pd.read_csv('Churn_Modelling.csv')
data = data.drop(columns=['RowNumber','CustomerId','Surname'])

label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])

one_hot_encoder_geo = OneHotEncoder(handle_unknown='ignore')
geo_encoded = one_hot_encoder_geo.fit_transform(data[['Geography']]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=one_hot_encoder_geo.get_feature_names_out(['Geography']))

data = pd.concat([data.drop('Geography',axis=1), geo_encoded_df], axis=1)

x = data.drop(columns=['Exited'])
y = data['Exited']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

# Save encoder and scaler for future use

with open('label_encoder_gender.pkl', 'wb') as le_file:
    pickle.dump(label_encoder_gender, le_file)

with open('one_hot_encoder_geo.pkl', 'wb') as ohe_file:
    pickle.dump(one_hot_encoder_geo, ohe_file)

with open('scaler.pkl', 'wb') as scaler_file:
    pickle.dump(scaler, scaler_file)


In [ ]:
# Define a function to create the model and try different parametere (KerasClassifier)

def create_model(hp):
    model = Sequential()
    model.add(Dense(
        units=hp.Int('units', min_value=16, max_value=128, step=16),
        activation=hp.Choice('activation', values=['relu', 'tanh']),
        input_dim=x_train.shape[1]
    ))
    for i in range(hp.Int('layers', 1, 3)):
        model.add(Dense(
            units=hp.Int(f'units_{i}', min_value=16, max_value=128, step=16),
            activation=hp.Choice(f'activation_{i}', values=['relu', 'tanh'])
        ))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(
        optimizer=hp.Choice('optimizer', values=['adam', 'rmsprop']),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model


In [ ]:
# KerasTuner ile hiperparametre araması
tuner = RandomSearch(
    create_model,
    objective='val_accuracy',
    max_trials=10,
    executions_per_trial=1,
    directory='my_dir',
    project_name='hyperparameter_tuning'
)


In [ ]:

# Hiperparametre araması
tuner.search(x_train, y_train, epochs=50, validation_split=0.2, verbose=1)
# En iyi modeli al
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

print(f"""
En iyi hiperparametreler:
- Units: {best_hps.get('units')}
- Activation: {best_hps.get('activation')}
- Optimizer: {best_hps.get('optimizer')}
""")

# En iyi modeli eğit
best_model = tuner.hypermodel.build(best_hps)
history = best_model.fit(x_train, y_train, epochs=50, validation_split=0.2, verbose=1)

# Test seti üzerinde değerlendirme
test_loss, test_acc = best_model.evaluate(x_test, y_test)
print(f"Test Accuracy: {test_acc}")

Trial 10 Complete [00h 00m 34s]
val_accuracy: 0.8618749976158142

Best val_accuracy So Far: 0.8643749952316284
Total elapsed time: 00h 04m 29s

En iyi hiperparametreler:
- Units: 96
- Activation: tanh
- Optimizer: rmsprop

Epoch 1/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7700 - loss: 0.4995 - val_accuracy: 0.8213 - val_loss: 0.4098
Epoch 2/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8204 - loss: 0.4172 - val_accuracy: 0.8394 - val_loss: 0.3809
Epoch 3/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8378 - loss: 0.3841 - val_accuracy: 0.8544 - val_loss: 0.3573
Epoch 4/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8475 - loss: 0.3659 - val_accuracy: 0.8575 - val_loss: 0.3466
Epoch 5/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8509 - loss: 0.3496 - val_accuracy: 0.8525 - val_loss: 0.3470
Epoch 6/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8557 - loss: 0.3460 - val_accuracy: 0.8631 - val_loss: 0.3427
Epoch 7/5

In [13]:
'''# Perform grid search
grid = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=-1, cv=3)
grid_result = grid.fit(x_train, y_train)

# Print the best parameters and score
print(f"Best: {grid_result.best_score_} using {grid_result.best_params_}")'''

'# Perform grid search\ngrid = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=-1, cv=3)\ngrid_result = grid.fit(x_train, y_train)\n\n# Print the best parameters and score\nprint(f"Best: {grid_result.best_score_} using {grid_result.best_params_}")'